## Task 1 -> Load and understand Dataset

In [1]:
import pandas as pd

In [2]:
# loading the dataset a=using pandas and storing it in df variable
df = pd.read_csv('book.csv')

# Now understanding the dataset 
print('Shape')
print(df.shape)

print('\nColumns Name :')
print(df.columns.to_list())

print("\nFirst 5 Rows:")
display(df.head(5))

print('Dataset info')
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

# Text Column used for recommendation
df['description']

Shape
(4766, 9)

Columns Name :
['Unnamed: 0', 'book_id', 'authors', 'original_publication_year', 'title', 'language_code', 'average_rating', 'image_url', 'description']

First 5 Rows:


,Unnamed: 0,book_id,authors,original_publication_year,title,language_code,average_rating,image_url,description
0,0,2767052,Suzanne Collins,2008.0,"The Hunger Games (The Hunger Games, #1)",eng,4.34,https://images.gr-assets.com/books/1447303603m...,First in the ground-breaking HUNGER GAMES tril...
1,1,3,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Sorcerer's Stone (Harry P...,eng,4.44,https://images.gr-assets.com/books/1474154022m...,Rescued from the outrageous neglect of his aun...
2,2,41865,Stephenie Meyer,2005.0,"Twilight (Twilight, #1)",en-US,3.57,https://images.gr-assets.com/books/1361039443m...,"When 17 year old Isabella Swan moves to Forks,..."
3,3,2657,Harper Lee,1960.0,To Kill a Mockingbird,eng,4.25,https://images.gr-assets.com/books/1361975680m...,Harper Lee's classic novel of a lawyer in the ...
4,4,4671,F. Scott Fitzgerald,1925.0,The Great Gatsby,eng,3.89,https://images.gr-assets.com/books/1490528560m...,The only authorized edition of the twentieth-c...


Dataset info
<class 'pandas.DataFrame'>
RangeIndex: 4766 entries, 0 to 4765
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Unnamed: 0                 4766 non-null   int64  
 1   book_id                    4766 non-null   int64  
 2   authors                    4766 non-null   str    
 3   original_publication_year  4766 non-null   float64
 4   title                      4766 non-null   str    
 5   language_code              4766 non-null   str    
 6   average_rating             4766 non-null   float64
 7   image_url                  4766 non-null   str    
 8   description                4766 non-null   str    
dtypes: float64(2), int64(2), str(5)
memory usage: 5.0 MB

Missing Values:
Unnamed: 0                   0
book_id                      0
authors                      0
original_publication_year    0
title                        0
language_code                0
average_rating 

0       First in the ground-breaking HUNGER GAMES tril...
1       Rescued from the outrageous neglect of his aun...
2       When 17 year old Isabella Swan moves to Forks,...
3       Harper Lee's classic novel of a lawyer in the ...
4       The only authorized edition of the twentieth-c...
                              ...                        
4761    He is mine and I am his. I will move every obs...
4762    SOME ARE BORN TO POWER SOME SEIZE IT AND SOME ...
4763    Picking up after the dramatic cliffhanger that...
4764    The Edge lies between worlds, on the border be...
4765    In Means of Ascent, Book Two of The Years of L...
Name: description, Length: 4766, dtype: str

In [3]:
# Remove unnecessary columns
df = df.drop(columns=[
    "Unnamed: 0",
    "book_id",
    "original_publication_year",
    "language_code"
])

## Task 2 — Text Preprocessing

In [4]:
import re
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation and special characters
    text = text = re.sub(r"[^a-zA-Z\s]", "", text)

    # Handle missing values if have
    df["description"] = df["description"].fillna("")

    # Removing stop words
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

# Applying preprocessing in description column
df['clean_text'] = df['description'].apply(preprocess_text)

# Display result
df[["description", "clean_text"]].head()
        

,description,clean_text
0,First in the ground-breaking HUNGER GAMES tril...,first groundbreaking hunger games trilogy set ...
1,Rescued from the outrageous neglect of his aun...,rescued outrageous neglect aunt uncle young bo...
2,"When 17 year old Isabella Swan moves to Forks,...",year old isabella swan moves forks washington ...
3,Harper Lee's classic novel of a lawyer in the ...,harper lees classic novel lawyer deep south de...
4,The only authorized edition of the twentieth-c...,authorized edition twentiethcentury classic fe...


## Task 3 -> TF-IDF Vectorization

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

tfidf_matrix = tfidf.fit_transform(df['clean_text'])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (4766, 5000)


## Task 4 -> Similarity Computation

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate cosine similarity between all books
similarity = cosine_similarity(tfidf_matrix)

# Similarity matrix shape
print("Cosine Similarity Shape:", similarity.shape)

# Display similarity scores for first book
print("\nSimilarity Scores for First Book:")
print(similarity[0])

Cosine Similarity Shape: (4766, 4766)

Similarity Scores for First Book:
[1.         0.         0.03869351 ... 0.         0.03022083 0.00999999]


## Task 5 -> Build Recommendation Function

In [13]:
# Creating function using title and its index to match with cosine similarity and getting its book name from index
def recommend_books(title):
    book_index = df[df['title'] == title].index[0]
    recommendation = similarity[book_index]
    book_list = sorted(enumerate(recommendation), reverse=True, key=lambda x: x[1])[1:6]
    for book in book_list:
        print(df.title[book[0]])

# Test
recommend_books(df["title"].iloc[17])

Fixed on You (Fixed, #1)
What Alice Forgot
Alice in the Country of Hearts, Vol. 01 (Alice in the Country of Hearts, #1)
Tales from a Not-So-Fabulous Life (Dork Diaries, #1)
Still Alice


In [8]:
recommend_books(df["title"].iloc[10])

The Transfer (Divergent, #0.1)
The Initiate (Divergent, #0.2)
The Traitor (Divergent, #0.4)
Free Four: Tobias Tells the Divergent Knife-Throwing Scene (Divergent, #1.5)
The World of Divergent: The Path to Allegiant (Divergent, #2.5)


In [9]:
recommend_books(df["title"].iloc[12])

The Namesake
Among the Hidden (Shadow Children, #1)
Buried Prey (Lucas Davenport, #21)
Blue-Eyed Devil (Travises, #2)
Girls in Love (Girls, #1)


## Task 6 -> Build Streamlit Application
Please refer folder for app.py

In [10]:
# Saving df in books.pickle to use in streamlit app
import pickle

with open('books.pickle', 'wb') as file:
    pickle.dump(df, file) 

In [12]:
import joblib

joblib.dump(tfidf, "tfidf.joblib")
joblib.dump(tfidf_matrix, "tfidf_matrix.joblib")

['tfidf_matrix.joblib']

## Task 7 -> Version Control with Git and Github
Created new repository and pushed the code

## Task 8 -> Deployment on Render
Created new Account and connected with github
Deployed streamlit app

## Task 9 -> Final Validation
Tested deplyed App

Recommendation is working correctly

Deployed URL -> "https://book-recommendation-system-ic8z.onrender.com/"